In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import pickle
from datetime import datetime

# ML imports
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

# XGBoost
import xgboost as xgb

warnings.filterwarnings('ignore')
print("Imports complete")

Imports complete


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
football_api_key = user_secrets.get_secret("MY_API_KEY")

In [3]:
# =============================================================================
# CELL 2: CONFIGURABLE PARAMETERS
# =============================================================================
# Adjust these parameters to experiment with different configurations

# ----- Data Parameters -----
ROLLING_WINDOW = 5          # Number of previous games for rolling stats (try: 3, 5, 10)
MIN_MATCHES_REQUIRED = 3    # Minimum matches before making predictions

# ----- Train/Test Split -----
TEST_SEASON = '2025-2026'     # Season to use as test set

# ----- XGBoost Parameters -----
XGBOOST_PARAMS = {
    'objective': 'count:poisson',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.001,
    'reg_lambda': 0.01,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}
# ----- Early Stopping -----
EARLY_STOPPING_ROUNDS = 50  # Stop if no improvement for N rounds
VALIDATION_SPLIT = 0.2     # Fraction of training data for validation

# ----- Over/Under Thresholds -----
OU_THRESHOLDS = [8.5, 9.5, 10.5, 11.5, 12.5]

print("Parameters configured:")
print(f"  Rolling window: {ROLLING_WINDOW}")
print(f"  Test season: {TEST_SEASON}")
print(f"  XGBoost max_depth: {XGBOOST_PARAMS['max_depth']}")
print(f"  XGBoost learning_rate: {XGBOOST_PARAMS['learning_rate']}")

Parameters configured:
  Rolling window: 5
  Test season: 2025-2026
  XGBoost max_depth: 6
  XGBoost learning_rate: 0.05


In [4]:
import requests
import pandas as pd

def get_upcoming_fixtures(api_key):
    # Endpoint lấy các trận đấu của Premier League (Mã giải đấu là 'PL' hoặc ID 2021)
    url = "https://api.football-data.org/v4/competitions/PL/matches?status=SCHEDULED"
    headers = { 'X-Auth-Token': api_key }
    
    try:
        response = requests.get(url, headers=headers)
        data = response.json()
        
        fixtures = []
        for match in data['matches']:
            fixtures.append({
                'Date': match['utcDate'],
                'HomeTeam': match['homeTeam']['shortName'], # Hoặc .name tùy theo mapping
                'AwayTeam': match['awayTeam']['shortName'],
                'Season': '2025-26',
                'FTR': None # Trận chưa đá nên kết quả để trống
            })
            
        df_fixtures = pd.DataFrame(fixtures)
        # Chuyển đổi Date sang datetime và bỏ phần giờ để khớp với CSV
        df_fixtures['Date'] = pd.to_datetime(df_fixtures['Date']).dt.tz_localize(None)
        
        return df_fixtures
    
    except Exception as e:
        print(f"❌ Lỗi khi lấy API: {e}")
        return pd.DataFrame()

In [5]:
# =============================================================================
# CELL 3: Load Data from Football-Data.co.uk
# =============================================================================
SEASONS = {
    # '2000-01': '0001',
    # '2001-02': '0102',
    # '2002-03': '0203',
    # '2003-04': '0304',
    # '2004-05': '0405',
    '2005-06': '0506',
    '2006-07': '0607',
    '2007-08': '0708',
    '2008-09': '0809',
    '2009-10': '0910',
    '2010-11': '1011',
    '2011-12': '1112',
    '2012-13': '1213',
    '2013-14': '1314',
    '2014-15': '1415',
    '2015-16': '1516',
    '2016-17': '1617',
    '2017-18': '1718',
    '2018-19': '1819',
    '2019-20': '1920',
    '2020-21': '2021',
    '2021-22': '2122',
    '2022-23': '2223',
    '2023-24': '2324',
    '2024-25': '2425',
    '2025-26': '2526'
}

BASE_URL = 'https://www.football-data.co.uk/mmz4281/{code}/E0.csv'

COLS = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
        'HS', 'AS', 'HY', 'AY', 'HST', 'AST', 'HC', 'AC', 'HR', 'AR', 'HF', 'AF' ]

print("Loading data from Football-Data.co.uk...\n")
dfs = []
for season_name, season_code in SEASONS.items():
    url = BASE_URL.format(code=season_code)
    try:
        df = pd.read_csv(url, encoding='utf-8')
        available_cols = [c for c in COLS if c in df.columns]
        df = df[available_cols].copy()
        df['Season'] = season_name
        print(f"  {season_name}: {len(df)} matches")
        dfs.append(df)
    except Exception as e:
        print(f"  {season_name}: Failed - {e}")

df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=['Date', 'HomeTeam'])
try:
    # Gọi hàm lấy Fixtures mà chúng ta vừa viết (đảm bảo hàm này đã được định nghĩa trước)
    df_upcoming = get_upcoming_fixtures(football_api_key) 
    
    if not df_upcoming.empty:
        # df_upcoming = df_upcoming.sort_values('Date')
        df_upcoming = df_upcoming.head(10)
        
        team_mapping = {
            'Nottingham': "Nott'm Forest", 'Brighton Hove': 'Brighton',
            'Leeds United': 'Leeds', 'Wolverhampton': 'Wolves'
            # Kiểm tra lại tên chuẩn trong CSV của bạn
        }
        df_upcoming['HomeTeam'] = df_upcoming['HomeTeam'].replace(team_mapping)
        df_upcoming['AwayTeam'] = df_upcoming['AwayTeam'].replace(team_mapping)
        
        # Nối vào df chính
        df = pd.concat([df, df_upcoming], ignore_index=True)
        print(f"\n✅ Added {len(df_upcoming)} upcoming fixtures from API.")
except Exception as e:
    print(f"\n⚠️ Could not add API fixtures: {e}")

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

print(f"\nTotal matches: {len(df)}")
print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")

Loading data from Football-Data.co.uk...

  2005-06: 380 matches
  2006-07: 380 matches
  2007-08: 380 matches
  2008-09: 380 matches
  2009-10: 380 matches
  2010-11: 380 matches
  2011-12: 380 matches
  2012-13: 380 matches
  2013-14: 380 matches
  2014-15: 381 matches
  2015-16: 380 matches
  2016-17: 380 matches
  2017-18: 380 matches
  2018-19: 380 matches
  2019-20: 380 matches
  2020-21: 380 matches
  2021-22: 380 matches
  2022-23: 380 matches
  2023-24: 380 matches
  2024-25: 380 matches
  2025-26: 329 matches

✅ Added 10 upcoming fixtures from API.

Total matches: 7939
Date range: 2005-08-13 to 2026-05-02


In [6]:
df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,HST,AST,HC,AC,HR,AR,HF,AF,Season
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,2.0,6.0,7.0,8.0,0.0,0.0,14.0,16.0,2005-06
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,5.0,5.0,8.0,6.0,0.0,0.0,15.0,14.0,2005-06
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,7.0,4.0,6.0,6.0,0.0,0.0,12.0,13.0,2005-06
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,8.0,3.0,3.0,6.0,0.0,0.0,13.0,11.0,2005-06
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,2.0,7.0,5.0,0.0,1.0,0.0,17.0,11.0,2005-06


In [7]:
df.tail(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,HST,AST,HC,AC,HR,AR,HF,AF,Season
7929,2026-04-24 19:00:00,Sunderland,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7930,2026-04-25 11:30:00,Fulham,Aston Villa,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7931,2026-04-25 14:00:00,Liverpool,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7932,2026-04-25 14:00:00,West Ham,Everton,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7933,2026-04-25 14:00:00,Wolves,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7934,2026-04-25 16:30:00,Arsenal,Newcastle,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7935,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7936,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7937,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7938,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26


# ROLLING FEATURE

In [8]:
def create_rolling_features(df, n_games=5):
    # Đảm bảo sắp xếp theo thời gian
    data = df.sort_values('Date').copy()
    
    # Danh sách các chỉ số thô cần tính rolling
    metrics = {
        'goals': ['FTHG', 'FTAG'],
        'shots': ['HS', 'AS'],
        'shots_ot': ['HST', 'AST'],
        'corners': ['HC', 'AC'],
        'fouls': ['HF', 'AF'],
        'yellows': ['HY', 'AY'],
        'reds': ['HR', 'AR']
    }

    # Khởi tạo các cột kết quả với NaN
    for side in ['Home', 'Away']:
        for m in metrics.keys():
            data[f'{side}_Avg_{m}_{n_games}'] = np.nan
            data[f'{side}_Avg_{m}_Conceded_{n_games}'] = np.nan

    all_teams = pd.concat([data['HomeTeam'], data['AwayTeam']]).unique()

    for team in all_teams:
        # Lọc các trận của đội này
        team_mask = (data['HomeTeam'] == team) | (data['AwayTeam'] == team)
        team_df = data[team_mask].copy()
        
        # 1. Tạo một DataFrame TẠM THỜI chỉ chứa các con số để tính Rolling
        # Không đưa cột 'Date' vào đây
        rows = []
        for idx, row in team_df.iterrows():
            is_home = row['HomeTeam'] == team
            stats = {
                'goals': row['FTHG'] if is_home else row['FTAG'],
                'goals_conceded': row['FTAG'] if is_home else row['FTHG'],
                'shots': row['HS'] if is_home else row['AS'],
                'shots_conceded': row['AS'] if is_home else row['HS'], # Thêm cái này
                'shots_ot': row['HST'] if is_home else row['AST'],
                'shots_ot_conceded': row['AST'] if is_home else row['HST'], # VÀ CÁI NÀY
                'corners': row['HC'] if is_home else row['AC'],
                'fouls': row['HF'] if is_home else row['AF'],
                'yellows': row['HY'] if is_home else row['AY'],
                'reds': row['HR'] if is_home else row['AR']
            }
            rows.append(stats)
        
        # Tạo df chỉ gồm số, reset index để tránh lỗi Date ở Index
        temp_stats_df = pd.DataFrame(rows, index=team_df.index)
        
        # 2. Tính Rolling trên dữ liệu THUẦN SỐ
        roll_df = temp_stats_df.shift(1).rolling(window=n_games, min_periods=1).mean()
        
        # 3. Gán ngược lại vào DataFrame chính dựa trên Index
       # 3. Gán ngược lại vào DataFrame chính dựa trên Index
        for idx in team_df.index:
            is_home = data.at[idx, 'HomeTeam'] == team
            prefix = 'Home' if is_home else 'Away'
            
            # Gán toàn bộ các chỉ số đã tính trong roll_df
            for col in temp_stats_df.columns:
                # col ở đây bao gồm: goals, goals_conceded, shots, shots_ot, corners, ...
                if 'conceded' in col:
                    # Gán các chỉ số phòng ngự: shots_ot_conceded, goals_conceded...
                    data.at[idx, f'{prefix}_Avg_{col}_{n_games}'] = roll_df.at[idx, col]
                else:
                    # Gán các chỉ số tấn công: shots, shots_ot, corners...
                    data.at[idx, f'{prefix}_Avg_{col}_{n_games}'] = roll_df.at[idx, col]
            
            # QUAN TRỌNG: Để tính Defense_Ratio, chúng ta cần shots_ot bị đối phương sút
            # Trong stats ở bước 1, bạn chưa tính 'shots_ot_conceded'. Hãy thêm nó vào bước 1!
    return data

full_df = create_rolling_features(df)

In [9]:
full_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Away_Avg_yellows_5,Away_Avg_yellows_Conceded_5,Away_Avg_reds_5,Away_Avg_reds_Conceded_5,Home_Avg_goals_conceded_5,Home_Avg_shots_conceded_5,Home_Avg_shots_ot_conceded_5,Away_Avg_goals_conceded_5,Away_Avg_shots_conceded_5,Away_Avg_shots_ot_conceded_5
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
full_df.tail()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Away_Avg_yellows_5,Away_Avg_yellows_Conceded_5,Away_Avg_reds_5,Away_Avg_reds_Conceded_5,Home_Avg_goals_conceded_5,Home_Avg_shots_conceded_5,Home_Avg_shots_ot_conceded_5,Away_Avg_goals_conceded_5,Away_Avg_shots_conceded_5,Away_Avg_shots_ot_conceded_5
7934,2026-04-25 16:30:00,Arsenal,Newcastle,NaN,NaN,None,NaN,NaN,NaN,NaN,...,2.0,NaN,0.2,NaN,1.0,10.4,3.40,1.40,15.2,4.60
7935,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,...,0.6,NaN,0.0,NaN,1.4,14.6,4.20,0.80,13.0,3.40
7936,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1.6,NaN,0.2,NaN,0.4,9.6,2.80,2.20,16.2,4.80
7937,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1.5,NaN,0.0,NaN,1.0,13.0,3.75,0.75,17.5,4.25
7938,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,2.6,NaN,0.0,NaN,1.5,15.5,4.50,0.80,11.2,4.40


# Month Sin, Month Cos

In [11]:
def smart_date_parser(date_str):
    if pd.isna(date_str):
        return pd.NaT
    # Các định dạng phổ biến trong dữ liệu Football-data
    for fmt in ('%d/%m/%y', '%d/%m/%Y', '%Y-%m-%d'):
        try:
            return pd.to_datetime(date_str, format=fmt)
        except (ValueError, TypeError):
            continue
    # Nếu không khớp cái nào, để Pandas tự đoán lần cuối
    return pd.to_datetime(date_str, errors='coerce')

# Áp dụng hàm cho cột Date
full_df['Date'] = full_df['Date'].apply(smart_date_parser)

# Sau đó nhớ sort lại để đảm bảo thứ tự các trận đấu
full_df = full_df.sort_values('Date').reset_index(drop=True)
full_df['Month'] = full_df['Date'].dt.month

In [12]:
import numpy as np

# --- BƯỚC 1: Xử lý Month sang Sin/Cos ---
# Giúp mô hình hiểu tính chu kỳ (Tháng 12 gần Tháng 1)
full_df['Month_Sin'] = np.sin(2 * np.pi * full_df['Month'] / 12)
full_df['Month_Cos'] = np.cos(2 * np.pi * full_df['Month'] / 12)

# --- BƯỚC 2: Tạo cột Season (Cần thiết để reset Matchweek mỗi năm) ---
# Mùa giải Ngoại hạng Anh thường bắt đầu từ tháng 8
full_df['Season'] = full_df['Date'].apply(lambda x: f"{x.year}-{x.year+1}" if x.month >= 8 else f"{x.year-1}-{x.year}")

# --- BƯỚC 3: Tính Matchweek tự động ---
def calculate_matchweek(df):
    df = df.sort_values(['Season', 'Date']).copy()
    
    # Dictionary lưu số trận đã đá của từng đội trong từng mùa
    counts = {} 
    mws = []
    
    for idx, row in df.iterrows():
        s = row['Season']
        h = row['HomeTeam']
        a = row['AwayTeam']
        
        if s not in counts:
            counts[s] = {}
            
        # Lấy số trận hiện tại của 2 đội, mặc định là 0
        h_played = counts[s].get(h, 0)
        a_played = counts[s].get(a, 0)
        
        # Matchweek là số trận lớn nhất của 2 đội + 1 (để xử lý cả trận đá bù)
        current_mw = max(h_played, a_played) + 1
        mws.append(current_mw)
        
        # Cập nhật số trận đã đá cho trận tiếp theo
        counts[s][h] = h_played + 1
        counts[s][a] = a_played + 1
        
    df['Matchweek'] = mws
    return df

full_df = calculate_matchweek(full_df)

# Ranking

In [13]:
def add_league_rank_features(df):
    # Sắp xếp theo mùa giải và thời gian
    df = df.sort_values(['Season', 'Date']).copy()
    
    # Lưu trữ trạng thái bảng xếp hạng: {Season: {Team: {'pts': 0, 'gd': 0, 'gs': 0}}}
    season_stats = {}
    
    home_ranks = []
    away_ranks = []

    for idx, row in df.iterrows():
        s = row['Season']
        h = row['HomeTeam']
        a = row['AwayTeam']
        
        if s not in season_stats:
            season_stats[s] = {}
            
        # Khởi tạo đội bóng nếu chưa có trong mùa này
        for team in [h, a]:
            if team not in season_stats[s]:
                season_stats[s][team] = {'pts': 0, 'gd': 0, 'gs': 0}
        
        # --- BƯỚC 1: Tính Rank TRƯỚC trận đấu ---
        # Chuyển stats hiện tại thành list để sort
        current_table = []
        for team, stats in season_stats[s].items():
            current_table.append({
                'team': team,
                'pts': stats['pts'],
                'gd': stats['gd'],
                'gs': stats['gs']
            })
        
        # Sort theo: Điểm > Hiệu số > Số bàn thắng
        current_table.sort(key=lambda x: (x['pts'], x['gd'], x['gs']), reverse=True)
        
        # Tạo map Team -> Vị trí (Rank)
        rank_map = {item['team']: i + 1 for i, item in enumerate(current_table)}
        
        # Gán rank cho trận hiện tại (nếu là trận đầu tiên của mùa thì mặc định là giữa bảng)
        home_ranks.append(rank_map.get(h, 10))
        away_ranks.append(rank_map.get(a, 10))
        
        # --- BƯỚC 2: Cập nhật kết quả SAU trận đấu để tính cho các vòng sau ---
        if pd.notna(row['FTR']):
            fthg, ftag = row['FTHG'], row['FTAG']
            
            # Cập nhật điểm
            if row['FTR'] == 'H':
                season_stats[s][h]['pts'] += 3
            elif row['FTR'] == 'A':
                season_stats[s][a]['pts'] += 3
            else:
                season_stats[s][h]['pts'] += 1
                season_stats[s][a]['pts'] += 1
            
            # Cập nhật hiệu số (GD) và bàn thắng (GS)
            season_stats[s][h]['gd'] += (fthg - ftag)
            season_stats[s][a]['gd'] += (ftag - fthg)
            season_stats[s][h]['gs'] += fthg
            season_stats[s][a]['gs'] += ftag
            
    df['Home_Rank'] = home_ranks
    df['Away_Rank'] = away_ranks
    df['Rank_Diff'] = df['Home_Rank'] - df['Away_Rank'] # Độ chênh lệch thứ hạng
    
    return df

# Thực hiện tính toán
full_df = add_league_rank_features(full_df)

# Kiểm tra thử kết quả
print(full_df[['Date', 'HomeTeam', 'AwayTeam', 'Matchweek', 'Home_Rank', 'Away_Rank']].tail(10))

                    Date    HomeTeam        AwayTeam  Matchweek  Home_Rank  \
7929 2026-04-24 19:00:00  Sunderland   Nott'm Forest         34         11   
7930 2026-04-25 11:30:00      Fulham     Aston Villa         34         12   
7931 2026-04-25 14:00:00   Liverpool  Crystal Palace         34          5   
7932 2026-04-25 14:00:00    West Ham         Everton         34         17   
7933 2026-04-25 14:00:00      Wolves       Tottenham         34         20   
7934 2026-04-25 16:30:00     Arsenal       Newcastle         34          1   
7935 2026-04-27 19:00:00  Man United       Brentford         34          3   
7936 2026-05-01 19:00:00       Leeds         Burnley         34         15   
7937 2026-05-02 14:00:00   Brentford        West Ham         35          7   
7938 2026-05-02 14:00:00   Newcastle        Brighton         35         14   

      Away_Rank  
7929         16  
7930          4  
7931         13  
7932         10  
7933         18  
7934         14  
7935          7

# SHOT AND DEFENSE 

In [14]:
# Đảm bảo tên cột khớp với output của hàm Rolling (thường là viết thường nếu bạn dùng dict stats)
# Giả sử hàm Rolling của bạn trả về các cột như: Home_Avg_shots_5, Home_Avg_shots_ot_5...

# 1. Shot Efficiency (Hiệu quả dứt điểm - Tấn công)
full_df['Home_Shot_Efficiency'] = full_df['Home_Avg_shots_ot_5'] / (full_df['Home_Avg_shots_5'] + 1e-5)
full_df['Away_Shot_Efficiency'] = full_df['Away_Avg_shots_ot_5'] / (full_df['Away_Avg_shots_5'] + 1e-5)

# 2. Defense Factor (Hệ số phòng ngự)
# Lưu ý: "Home_Avg_goals_conceded_5" là số bàn thua
# Chúng ta cần chia cho số cú sút trúng đích mà ĐỐI THỦ gây ra (đã được tính trong hàm rolling)
full_df['Home_Defense_Ratio'] = full_df['Home_Avg_goals_conceded_5'] / (full_df['Home_Avg_shots_ot_conceded_5'] + 1e-5)
full_df['Away_Defense_Ratio'] = full_df['Away_Avg_goals_conceded_5'] / (full_df['Away_Avg_shots_ot_conceded_5'] + 1e-5)

# 3. Pressure Index (Chỉ số áp lực - Kết hợp Phạt góc)
full_df['Home_Attack_Pressure'] = (full_df['Home_Avg_shots_ot_5'] * 0.7) + (full_df['Home_Avg_corners_5'] * 0.3)
full_df['Away_Attack_Pressure'] = (full_df['Away_Avg_shots_ot_5'] * 0.7) + (full_df['Away_Avg_corners_5'] * 0.3)

# H2H HISTORY

In [15]:
def add_h2h_features(df):
    h2h_map = {} 
    
    h2h_home_wins = []
    h2h_away_wins = []
    h2h_draws = []
    
    for idx, row in df.iterrows():
        # Đảm bảo tên đội luôn được sắp xếp để tạo key duy nhất cho cặp đối đầu
        teams = tuple(sorted([row['HomeTeam'], row['AwayTeam']]))
        team_h = row['HomeTeam']
        team_a = row['AwayTeam']
        
        # Khởi tạo nếu cặp đội này lần đầu xuất hiện
        if teams not in h2h_map:
            h2h_map[teams] = {team_h: 0, team_a: 0, 'draw': 0}
        # Trường hợp đặc biệt: Đội mới thăng hạng có thể chưa có key trong dict con
        if team_h not in h2h_map[teams]: h2h_map[teams][team_h] = 0
        if team_a not in h2h_map[teams]: h2h_map[teams][team_a] = 0
        
        # 1. LẤY DỮ LIỆU LỊCH SỬ (Trước khi trận này diễn ra)
        # Đây là dữ liệu Model sẽ dùng để dự đoán
        current_h2h = h2h_map[teams]
        h2h_home_wins.append(current_h2h.get(team_h, 0))
        h2h_away_wins.append(current_h2h.get(team_a, 0))
        h2h_draws.append(current_h2h['draw'])
        
        # 2. CHỈ CẬP NHẬT KẾT QUẢ NẾU TRẬN ĐẤU ĐÃ CÓ KẾT QUẢ
        # Dùng pd.notna() để bỏ qua các trận tuần tới (FTR = NaN)
        if pd.notna(row['FTR']):
            if row['FTR'] == 'H':
                h2h_map[teams][team_h] += 1
            elif row['FTR'] == 'A':
                h2h_map[teams][team_a] += 1
            elif row['FTR'] == 'D': # Ghi rõ 'D' cho chắc chắn
                h2h_map[teams]['draw'] += 1
            
    df['Home_H2H_Wins'] = h2h_home_wins
    df['Away_H2H_Wins'] = h2h_away_wins
    df['H2H_Draws'] = h2h_draws
    return df
full_df = add_h2h_features(full_df)

In [16]:
full_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Rank_Diff,Home_Shot_Efficiency,Away_Shot_Efficiency,Home_Defense_Ratio,Away_Defense_Ratio,Home_Attack_Pressure,Away_Attack_Pressure,Home_H2H_Wins,Away_H2H_Wins,H2H_Draws
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [17]:
full_df.tail(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Rank_Diff,Home_Shot_Efficiency,Away_Shot_Efficiency,Home_Defense_Ratio,Away_Defense_Ratio,Home_Attack_Pressure,Away_Attack_Pressure,Home_H2H_Wins,Away_H2H_Wins,H2H_Draws
7929,2026-04-24 19:00:00,Sunderland,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-5,0.368421,0.396226,0.230769,0.222222,4.200,4.140,1,0,0
7930,2026-04-25 11:30:00,Fulham,Aston Villa,NaN,NaN,None,NaN,NaN,NaN,NaN,...,8,0.214286,0.367647,0.210526,0.423076,3.840,4.820,6,12,9
7931,2026-04-25 14:00:00,Liverpool,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-8,0.337500,0.265306,0.240000,0.160000,5.820,2.900,15,6,4
7932,2026-04-25 14:00:00,West Ham,Everton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,7,0.347826,0.491228,0.136363,0.260869,3.620,4.880,10,18,11
7933,2026-04-25 14:00:00,Wolves,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,2,0.297872,0.428571,0.357142,0.499999,2.860,5.940,9,7,5
7934,2026-04-25 16:30:00,Arsenal,Newcastle,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-13,0.294117,0.370370,0.294117,0.304347,4.660,3.880,25,6,6
7935,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-4,0.382353,0.294117,0.333333,0.235293,5.440,3.120,5,3,1
7936,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-4,0.310811,0.395348,0.142857,0.458332,4.840,3.940,3,1,1
7937,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-10,0.282608,0.351351,0.266666,0.176470,3.325,3.625,7,1,1
7938,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,5,0.357143,0.370967,0.333333,0.181818,3.825,4.300,2,7,8


# ELO

In [18]:
import pandas as pd
import numpy as np

def calculate_elo(df, k_factor=30, home_advantage=100):
    # 1. Khởi tạo dictionary lưu điểm ELO hiện tại của các đội
    # Mặc định mỗi đội là 1500
    all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()
    elo_ratings = {team: 1500 for team in all_teams}
    
    # Danh sách để lưu ELO TRƯỚC trận đấu (để đưa vào Feature)
    home_elo_before = []
    away_elo_before = []
    
    # Duyệt qua từng trận đấu theo thứ tự thời gian
    for idx, row in df.iterrows():
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']
        result = row['FTR']
        
        # Lấy điểm ELO hiện tại
        r_h = elo_ratings[home_team]
        r_a = elo_ratings[away_team]
        
        # Lưu lại điểm TRƯỚC trận để làm Feature dự đoán
        home_elo_before.append(r_h)
        away_elo_before.append(r_a)
        
        # 2. Tính toán xác suất mong đợi (Expected Score)
        # Có tính đến lợi thế sân nhà (home_advantage)
        e_h = 1 / (1 + 10 ** ((r_a - (r_h + home_advantage)) / 400))
        e_a = 1 - e_h
        
        # 3. Xác định kết quả thực tế (Actual Score)
        if result == 'H':
            s_h, s_a = 1, 0
        elif result == 'A':
            s_h, s_a = 0, 1
        else: # Hòa
            s_h, s_a = 0.5, 0.5
            
        # 4. Cập nhật điểm ELO sau trận đấu
        elo_ratings[home_team] = r_h + k_factor * (s_h - e_h)
        elo_ratings[away_team] = r_a + k_factor * (s_a - e_a)
        
    # Thêm vào dataframe chính
    df['Home_Elo'] = home_elo_before
    df['Away_Elo'] = away_elo_before
    df['Elo_Diff'] = df['Home_Elo'] - df['Away_Elo']
    
    return df, elo_ratings

# Chạy hàm
full_df, final_elo = calculate_elo(full_df)

# Kiểm tra thử Top 5 đội có ELO cao nhất hiện tại (cuối dữ liệu)
print("Top đội mạnh nhất dựa trên ELO:")
sorted_elo = sorted(final_elo.items(), key=lambda x: x[1], reverse=True)
print(sorted_elo[:5])

Top đội mạnh nhất dựa trên ELO:
[('Man City', 1814.6250587821653), ('Arsenal', 1789.00874830869), ('Liverpool', 1705.18700681661), ('Aston Villa', 1694.0872808106478), ('Man United', 1677.253965025819)]


# AGGRESSION

In [19]:
def calculate_advanced_aggression(df):
    data = df.copy()
    
    # 1. Tính điểm Aggression (Chỉ những trận đã có dữ liệu thẻ phạt)
    # Dùng fillna(0) để các trận tương lai không làm hỏng phép tính mean() sau này
    data['Home_Points'] = (data['HF'].fillna(0) * 0.5) + data['HY'].fillna(0) + (data['HR'].fillna(0) * 3)
    data['Away_Points'] = (data['AF'].fillna(0) * 0.5) + data['AY'].fillna(0) + (data['AR'].fillna(0) * 3)
    data['Total_Match_Points'] = data['Home_Points'] + data['Away_Points']

    # Tính giá trị trung bình toàn giải (chỉ tính trên các trận đã đá)
    # Lấy những trận mà FTR không phải None/NaN
    past_matches = data[data['FTR'].notna()]
    global_mean_total = past_matches['Total_Match_Points'].mean()
    global_mean_home = past_matches['Home_Points'].mean()
    global_mean_away = past_matches['Away_Points'].mean()

    def get_features(row):
        home, away, date = row['HomeTeam'], row['AwayTeam'], row['Date']
        
        # --- Phần 1: H2H Rivalry ---
        h2h = data[
            ((data['HomeTeam'] == home) & (data['AwayTeam'] == away) |
             (data['HomeTeam'] == away) & (data['AwayTeam'] == home)) &
            (data['Date'] < date)
        ].sort_values('Date', ascending=False).head(5)
        
        # Nếu không có H2H, dùng trung bình toàn giải của các trận ĐÃ ĐÁ
        h2h_idx = h2h['Total_Match_Points'].mean() if not h2h.empty else global_mean_total

        # --- Phần 2: Team Style ---
        def calc_team_pts(team, current_date, default_val):
            recent = data[
                ((data['HomeTeam'] == team) | (data['AwayTeam'] == team)) & 
                (data['Date'] < current_date) &
                (data['FTR'].notna()) # QUAN TRỌNG: Chỉ lấy phong độ từ các trận đã kết thúc
            ].sort_values('Date', ascending=False).head(5)
            
            pts = []
            for _, r in recent.iterrows():
                pts.append(r['Home_Points'] if r['HomeTeam'] == team else r['Away_Points'])
            return np.mean(pts) if pts else default_val

        home_style = calc_team_pts(home, date, global_mean_home)
        away_style = calc_team_pts(away, date, global_mean_away)

        return pd.Series([h2h_idx, home_style, away_style])

    # Thực hiện apply
    data[['H2H_Rivalry', 'Home_Style_Agg', 'Away_Style_Agg']] = data.apply(get_features, axis=1)
    return data

In [20]:
full_df = calculate_advanced_aggression(full_df)

In [21]:
full_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,H2H_Draws,Home_Elo,Away_Elo,Elo_Diff,Home_Points,Away_Points,Total_Match_Points,H2H_Rivalry,Home_Style_Agg,Away_Style_Agg
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,...,0,1500.0,1500.0,0.0,7.0,10.0,17.0,14.894564,7.162694,7.73187
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,...,0,1500.0,1500.0,0.0,10.5,8.0,18.5,14.894564,7.162694,7.73187
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,...,0,1500.0,1500.0,0.0,7.0,8.5,15.5,14.894564,7.162694,7.73187
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,...,0,1500.0,1500.0,0.0,8.5,8.5,17.0,14.894564,7.162694,7.73187
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,...,0,1500.0,1500.0,0.0,13.5,8.5,22.0,14.894564,7.162694,7.73187


In [22]:
full_df.tail(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,H2H_Draws,Home_Elo,Away_Elo,Elo_Diff,Home_Points,Away_Points,Total_Match_Points,H2H_Rivalry,Home_Style_Agg,Away_Style_Agg
7929,2026-04-24 19:00:00,Sunderland,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,...,0,1554.981935,1574.161672,-19.179737,0.0,0.0,0.0,14.5,10.2,7.0
7930,2026-04-25 11:30:00,Fulham,Aston Villa,NaN,NaN,None,NaN,NaN,NaN,NaN,...,9,1598.717108,1693.878390,-95.161282,0.0,0.0,0.0,17.5,6.3,5.5
7931,2026-04-25 14:00:00,Liverpool,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,...,4,1713.402341,1599.703765,113.698575,0.0,0.0,0.0,16.3,6.6,9.0
7932,2026-04-25 14:00:00,West Ham,Everton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,11,1552.302039,1628.498302,-76.196262,0.0,0.0,0.0,13.7,7.6,5.1
7933,2026-04-25 14:00:00,Wolves,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,5,1470.307459,1470.025945,0.281514,0.0,0.0,0.0,15.9,8.1,8.9
7934,2026-04-25 16:30:00,Arsenal,Newcastle,NaN,NaN,None,NaN,NaN,NaN,NaN,...,6,1799.716810,1588.750456,210.966355,0.0,0.0,0.0,16.1,6.8,8.2
7935,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1683.471410,1630.242527,53.228883,0.0,0.0,0.0,14.7,10.0,5.1
7936,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1569.397183,1402.325464,167.071719,0.0,0.0,0.0,13.5,7.8,7.3
7937,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1636.459972,1551.275954,85.184017,0.0,0.0,0.0,13.0,5.1,7.6
7938,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,8,1599.458517,1656.436679,-56.978161,0.0,0.0,0.0,16.5,8.2,8.8


In [23]:
import pandas as pd
import numpy as np

def compute_advanced_corner_features(df, n=5):
    # 1. Chuẩn hóa ngày tháng và sắp xếp
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Target thực tế (để kiểm tra tương quan nếu cần)
    df['TotalCorners'] = df['HC'] + df['AC']
    
    # Xác định các trận đã đá (có kết quả) để làm dữ liệu lịch sử
    valid_mask = df['HC'].notna() & df['AC'].notna() & df['HS'].notna()
    
    # 2. Khởi tạo các cột mới
    new_cols = [
        'home_corners_for', 'home_corners_against', 'home_corners_total', 'home_corner_std',
        'away_corners_for', 'away_corners_against', 'away_corners_total', 'away_corner_std',
        'home_shots_for', 'home_shots_against', 'home_sot_for', 'home_sot_against',
        'away_shots_for', 'away_shots_against', 'away_sot_for', 'away_sot_against',
        'home_blocked_shots', 'home_blocked_against', 'away_blocked_shots', 'away_blocked_against',
        'home_shot_dominance', 'away_shot_dominance'
    ]
    for col in new_cols: df[col] = np.nan

    all_teams = set(df['HomeTeam'].unique()) | set(df['AwayTeam'].unique())

    # 3. Tính toán Venue-Aware Rolling (Phong độ theo sân)
    for team in all_teams:
        # Xử lý cho Team khi đá SÂN NHÀ
        h_idx = df[df['HomeTeam'] == team].index.tolist()
        h_history = [i for i in h_idx if valid_mask[i]]
        for idx in h_idx:
            prev = [i for i in h_history if df.loc[i, 'Date'] < df.loc[idx, 'Date']][-n:]
            if len(prev) >= n:
                data = df.loc[prev]
                df.loc[idx, 'home_corners_for'] = data['HC'].mean()
                df.loc[idx, 'home_corners_against'] = data['AC'].mean()
                df.loc[idx, 'home_corners_total'] = (data['HC'] + data['AC']).mean()
                df.loc[idx, 'home_corner_std'] = data['HC'].std()
                df.loc[idx, 'home_shots_for'] = data['HS'].mean()
                df.loc[idx, 'home_shots_against'] = data['AS'].mean()
                df.loc[idx, 'home_sot_for'] = data['HST'].mean()
                df.loc[idx, 'home_blocked_shots'] = (data['HS'] - data['HST']).mean()
                df.loc[idx, 'home_shot_dominance'] = (data['HS'] - data['AS']).mean()

        # Xử lý cho Team khi đá SÂN KHÁCH
        a_idx = df[df['AwayTeam'] == team].index.tolist()
        a_history = [i for i in a_idx if valid_mask[i]]
        for idx in a_idx:
            prev = [i for i in a_history if df.loc[i, 'Date'] < df.loc[idx, 'Date']][-n:]
            if len(prev) >= n:
                data = df.loc[prev]
                df.loc[idx, 'away_corners_for'] = data['AC'].mean()
                df.loc[idx, 'away_corners_against'] = data['HC'].mean()
                df.loc[idx, 'away_corners_total'] = (data['HC'] + data['AC']).mean()
                df.loc[idx, 'away_corner_std'] = data['AC'].std()
                df.loc[idx, 'away_shots_for'] = data['AS'].mean()
                df.loc[idx, 'away_shots_against'] = data['HS'].mean()
                df.loc[idx, 'away_sot_for'] = data['AST'].mean()
                df.loc[idx, 'away_blocked_shots'] = (data['AS'] - data['AST']).mean()
                df.loc[idx, 'away_shot_dominance'] = (data['AS'] - data['HS']).mean()

    # 4. Tính toán các chỉ số Match-level (Chỉ số tổng hợp trận đấu)
    # Expected Corners (Kỳ vọng phạt góc)
    df['expected_corners_total'] = (df['home_corners_total'] + df['away_corners_total']) / 2
    
    # Pressure Index (Chỉ số áp lực dựa trên tỷ lệ sút)
    df['home_shot_share'] = df['home_shots_for'] / (df['home_shots_for'] + df['home_shots_against']).replace(0, 1)
    df['away_shot_share'] = df['away_shots_for'] / (df['away_shots_for'] + df['away_shots_against']).replace(0, 1)
    df['pressure_sum'] = df['home_shot_share'] + df['away_shot_share']
    
    # Efficiency (Số phạt góc kiếm được trên mỗi cú sút)
    df['home_corners_per_shot'] = df['home_corners_for'] / df['home_shots_for'].replace(0, 1)
    df['away_corners_per_shot'] = df['away_corners_for'] / df['away_shots_for'].replace(0, 1)
    df['combined_corners_per_shot'] = df['home_corners_per_shot'] + df['away_corners_per_shot']
    
    # Volatility (Độ biến động - CV càng cao thì càng khó đoán)
    df['combined_corner_volatility'] = df['home_corner_std'] + df['away_corner_std']

    return df


full_df = compute_advanced_corner_features(full_df)

In [24]:
full_df.to_csv('/kaggle/working/epl_full_ft.csv')

### FEATURE COLUMNS

In [25]:
FEATURE_COLUMNS = [
    'Month_Sin', 'Month_Cos',
    'Matchweek',
    'Home_Rank', 'Away_Rank', 'Rank_Diff',
    'Home_Shot_Efficiency', 'Away_Shot_Efficiency',
    'Home_Defense_Ratio', 'Away_Defense_Ratio',
    'Home_Attack_Pressure', 'Away_Attack_Pressure',
    'Home_H2H_Wins', 'Away_H2H_Wins', 'H2H_Draws',
    'Home_Elo', 'Away_Elo', 'Elo_Diff',
    'H2H_Rivalry', 'Home_Style_Agg', 'Away_Style_Agg',

    'home_corners_for', 'home_corners_against', 'home_corners_total', 'home_corner_std',
    'away_corners_for', 'away_corners_against', 'away_corners_total', 'away_corner_std',
    'home_shots_for', 'home_shots_against', 'home_sot_for', 
    'away_shots_for', 'away_shots_against', 'away_sot_for',
    'home_blocked_shots', 'away_blocked_shots', 
    'home_shot_dominance', 'away_shot_dominance', 'expected_corners_total', 
    'home_shot_share', 'away_shot_share', 'pressure_sum', 
    'home_corners_per_shot', 'away_corners_per_shot', 
    'combined_corners_per_shot', 'combined_corner_volatility'
]

TARGET_COLUMNS = ['FTHG','FTAG','FTR','HS','AS','HY','AY','HST','AST','HC','AC','HR', 'AR','HF','AF']
TARGET_COLUMN = 'FTR'

TARGET_HOME = 'HY'
TARGET_AWAY = 'AY'

# Train/Test SPLIT

In [26]:

# Filter to rows with complete features
df_model = full_df.dropna(subset=FEATURE_COLUMNS).copy()
print(f"Matches with complete features: {len(df_model)} / {len(full_df)}")

for col in FEATURE_COLUMNS:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

df_model[FEATURE_COLUMNS] = df_model[FEATURE_COLUMNS].fillna(0)

# Time-based split
train_df = df_model[df_model['Season'] != TEST_SEASON].copy()
test_df = df_model[df_model['Season'] == TEST_SEASON].copy()

print(f"\nTraining set: {len(train_df)} matches")
print(f"  Seasons: {train_df['Season'].unique().tolist()}")
print(f"\nTest set: {len(test_df)} matches")
print(f"  Season: {TEST_SEASON}")

# Prepare X and y
X_train = train_df[FEATURE_COLUMNS]
y_train_home = train_df[TARGET_HOME]
y_train_away = train_df[TARGET_AWAY]

X_test = test_df[FEATURE_COLUMNS]
y_test_home = test_df[TARGET_HOME]
y_test_away = test_df[TARGET_AWAY]


Matches with complete features: 7598 / 7939

Training set: 7259 matches
  Seasons: ['2005-2006', '2006-2007', '2007-2008', '2008-2009', '2009-2010', '2010-2011', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']

Test set: 339 matches
  Season: 2025-2026


In [27]:
# =============================================================================
# CELL 8: Create Validation Split for Early Stopping
# =============================================================================

# Use last portion of training data as validation (time-based)
val_size = int(len(X_train) * VALIDATION_SPLIT)

X_train_fit = X_train.iloc[:-val_size]
y_train_home_fit = y_train_home.iloc[:-val_size]
y_train_away_fit = y_train_away.iloc[:-val_size]

X_val = X_train.iloc[-val_size:]
y_val_home = y_train_home.iloc[-val_size:]
y_val_away = y_train_away.iloc[-val_size:]



In [28]:
print("🚀 Training XGBoost Classifier...")
print(f"Parameters: {XGBOOST_PARAMS}")

# 1. Đảm bảo dữ liệu truyền vào chỉ gồm các Features đã chọn và là kiểu số
X_train_final = X_train_fit[FEATURE_COLUMNS].astype('float32')
X_val_final = X_val[FEATURE_COLUMNS].astype('float32')

# 2. Khởi tạo model
model_home = xgb.XGBRegressor(
    **XGBOOST_PARAMS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS
)

model_away = xgb.XGBRegressor(
    **XGBOOST_PARAMS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS
)

# 3. Train với eval set
model_home.fit(
    X_train_final, y_train_home_fit,
    eval_set=[(X_val_final, y_val_home)],
    verbose=50,
# Hiển thị tiến trình mỗi 50 vòng
)

model_away.fit(
    X_train_final, y_train_away_fit,
    eval_set=[(X_val_final, y_val_away)],
    verbose=50,
# Hiển thị tiến trình mỗi 50 vòng
)

print(f"\n✅ Training complete!")

print(f"\nTraining complete!")
print(f"\nHome")
print(f"Best iteration: {model_home.best_iteration}")
print(f"Best validation RMSE: {model_home.best_score:.4f}")

print(f"\nAway")
print(f"Best iteration: {model_away.best_iteration}")
print(f"Best validation RMSE: {model_away.best_score:.4f}")

🚀 Training XGBoost Classifier...
Parameters: {'objective': 'count:poisson', 'eval_metric': 'rmse', 'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_weight': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.001, 'reg_lambda': 0.01, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
[0]	validation_0-rmse:1.36501
[50]	validation_0-rmse:1.33849
[100]	validation_0-rmse:1.33463
[141]	validation_0-rmse:1.33968
[0]	validation_0-rmse:1.39764
[50]	validation_0-rmse:1.38655
[100]	validation_0-rmse:1.38693
[113]	validation_0-rmse:1.38887

✅ Training complete!

Training complete!

Home
Best iteration: 91
Best validation RMSE: 1.3344

Away
Best iteration: 63
Best validation RMSE: 1.3852


In [29]:
import lightgbm as lgb

LIGHTGBM_PARAMS = {
    'objective': 'regression',      # Mục tiêu là dự báo số (RMSE)
    'metric': 'rmse',               # Chỉ số đo lường lỗi
    'verbosity': -1,                # Giảm bớt các thông báo thừa
    'boosting_type': 'gbdt',        # Thuật toán chính của LightGBM
    'random_state': 63,
    
    # Các thông số quan trọng để tránh Overfitting
    'learning_rate': 0.005,          # Tốc độ học (thường từ 0.01 đến 0.1)
    'num_leaves': 31,               # Số lá tối đa trong 1 cây (rất quan trọng)
    'max_depth': -1,                # -1 là không giới hạn chiều sâu
    'min_child_samples': 10,        # Số lượng dữ liệu tối thiểu ở mỗi lá
    'feature_fraction': 0.8,        # Tương đương colsample_bytree của XGBoost
    'bagging_fraction': 0.8,        # Tương đương subsample của XGBoost
    'bagging_freq': 5,              # Thực hiện bagging sau mỗi 5 vòng
    'reg_alpha': 0.1,               # L1 regularization
    'reg_lambda': 0.1,              # L2 regularization
}

EARLY_STOPPING_ROUNDS = 100         # Số vòng chờ nếu lỗi không giảm

print("🚀 Training LightGBM Regressor...")
# Đảm bảo bạn đã định nghĩa LIGHTGBM_PARAMS (thường tương tự XGBoost nhưng tên một số tham số có thể khác)
print(f"Parameters: {LIGHTGBM_PARAMS}")

# 1. Đảm bảo dữ liệu truyền vào là kiểu số (LGBM xử lý tốt float32)
X_train_final = X_train_fit[FEATURE_COLUMNS].astype('float32')
X_val_final = X_val[FEATURE_COLUMNS].astype('float32')

# 2. Khởi tạo model với giao diện Scikit-learn
# Lưu ý: LightGBM dùng 'n_estimators' thay vì 'num_boost_round' trong wrapper này
model_home = lgb.LGBMRegressor(
    **LIGHTGBM_PARAMS,
    n_estimators=10000, # Số vòng tối đa, sẽ dừng sớm nhờ early_stopping
    importance_type='gain'
)

model_away = lgb.LGBMRegressor(
    **LIGHTGBM_PARAMS,
    n_estimators=10000,
    importance_type='gain'
)

# 3. Train với Early Stopping
# Trong LightGBM mới, early_stopping và log được đưa vào callbacks
callbacks = [
    lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
    lgb.log_evaluation(period=50) # Hiển thị tiến trình mỗi 50 vòng
]

model_home.fit(
    X_train_final, y_train_home_fit,
    eval_set=[(X_val_final, y_val_home)],
    eval_metric='rmse',
    callbacks=callbacks
)

model_away.fit(
    X_train_final, y_train_away_fit,
    eval_set=[(X_val_final, y_val_away)],
    eval_metric='rmse',
    callbacks=callbacks
)

print(f"\n✅ Training complete!")

print(f"\nHome")
print(f"Best iteration: {model_home.best_iteration_}")
print(f"Best validation RMSE: {model_home.best_score_['valid_0']['rmse']:.4f}")

print(f"\nAway")
print(f"Best iteration: {model_away.best_iteration_}")
print(f"Best validation RMSE: {model_away.best_score_['valid_0']['rmse']:.4f}")


🚀 Training LightGBM Regressor...
Parameters: {'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'boosting_type': 'gbdt', 'random_state': 63, 'learning_rate': 0.005, 'num_leaves': 31, 'max_depth': -1, 'min_child_samples': 10, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.1, 'reg_lambda': 0.1}
Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 1.35523
[100]	valid_0's rmse: 1.34739
[150]	valid_0's rmse: 1.34255
[200]	valid_0's rmse: 1.33892
[250]	valid_0's rmse: 1.33608
[300]	valid_0's rmse: 1.33514
[350]	valid_0's rmse: 1.33426
[400]	valid_0's rmse: 1.33327
[450]	valid_0's rmse: 1.3327
[500]	valid_0's rmse: 1.3322
[550]	valid_0's rmse: 1.33227
[600]	valid_0's rmse: 1.3325
Early stopping, best iteration is:
[509]	valid_0's rmse: 1.33201
Training until validation scores don't improve for 100 rounds
[50]	valid_0's rmse: 1.39436
[100]	valid_0's rmse: 1.39038
[150]	valid_0's rmse: 1.3881
[200]	valid_0's rmse: 1.

In [30]:
!pip install tabpfn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.4/227.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-ml-py
    Found existing installation: nvidia-ml-py 13.590.44
    Uninstalling nvidia-ml-py-13.590.44:
      Successfully uninstalled nvidia-ml-py-13.590.44
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whic

In [31]:
!pip install tabpfn-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.6 MB/s eta 0:00:00


In [ ]:

from tabpfn_client import TabPFNClassifier, set_access_token, TabPFNRegressor

# set_access_token("")

print("🚀 Training with TabPFN Regressor...")

# TabPFN không cần train lâu, nó 'học' ngay khi bạn gọi fit
model_home_pfn = TabPFNRegressor() # Dùng 'cuda' nếu có GPU
model_away_pfn = TabPFNRegressor()

# 1. Fit mô hình (thực tế là nạp dữ liệu vào context)
# Lưu ý: TabPFN hoạt động tốt nhất khi tập Train < 10,000 mẫu
model_home_pfn.fit(X_train_final, y_train_home_fit)
model_away_pfn.fit(X_train_final, y_train_away_fit)

# 2. Dự đoán
y_pred_home = model_home_pfn.predict(X_val_final)
y_pred_away = model_away_pfn.predict(X_val_final)

# Tính toán RMSE để so sánh với LightGBM
from sklearn.metrics import mean_squared_error
import numpy as np

rmse_home = np.sqrt(mean_squared_error(y_val_home, y_pred_home))
rmse_away = np.sqrt(mean_squared_error(y_val_away, y_pred_away))

print(f"TabPFN Home RMSE: {rmse_home:.4f}")
print(f"TabPFN Away RMSE: {rmse_away:.4f}")

🚀 Training with TabPFN Regressor...


Processing: 100%|██████████| [00:01<00:00]
Processing: 100%|██████████| [00:01<00:00]

TabPFN Home RMSE: 1.3441
TabPFN Away RMSE: 1.4030


In [33]:
# Chuẩn bị test data
X_test_final = X_test[FEATURE_COLUMNS].astype('float32')

# Predict goals
home_goals_pred = model_home.predict(X_test_final)
away_goals_pred = model_away.predict(X_test_final)

# Clip cho realistic
import numpy as np
home_goals_pred = np.clip(home_goals_pred, 0, 5)
away_goals_pred = np.clip(away_goals_pred, 0, 5)

In [34]:
X_test_final = X_test[FEATURE_COLUMNS].astype('float32')

home_lambda = model_home.predict(X_test_final)
away_lambda = model_away.predict(X_test_final)

import numpy as np
home_lambda = np.clip(home_lambda, 0, 5)
away_lambda = np.clip(away_lambda, 0, 5)

df_goals = pd.DataFrame({
    'xG_Home': home_lambda,
    'xG_Away': away_lambda
})

# Round cho đẹp
df_goals['xG_Home'] = df_goals['xG_Home'].round(2)
df_goals['xG_Away'] = df_goals['xG_Away'].round(2)

# Predicted score
df_goals['Pred_Score'] = (
    df_goals['xG_Home'].astype(str) + ' - ' + df_goals['xG_Away'].astype(str)
)

teams_info = test_df[['Date', 'HomeTeam', 'AwayTeam', 'Home_Rank', 'Away_Rank', 'Matchweek']].reset_index(drop=True)

final_display = pd.concat([teams_info, df_goals], axis=1)

print(f"✅ Đã xử lý xong {len(final_display)} trận cuối mùa {TEST_SEASON}.")

display(final_display.tail(10))

✅ Đã xử lý xong 339 trận cuối mùa 2025-2026.


,Date,HomeTeam,AwayTeam,Home_Rank,Away_Rank,Matchweek,xG_Home,xG_Away,Pred_Score
329,2026-04-24 19:00:00,Sunderland,Nott'm Forest,11,16,34,1.56,1.67,1.56 - 1.67
330,2026-04-25 11:30:00,Fulham,Aston Villa,12,4,34,1.68,1.88,1.68 - 1.88
331,2026-04-25 14:00:00,Liverpool,Crystal Palace,5,13,34,1.30,1.78,1.3 - 1.78
332,2026-04-25 14:00:00,West Ham,Everton,17,10,34,1.78,1.56,1.78 - 1.56
333,2026-04-25 14:00:00,Wolves,Tottenham,20,18,34,1.66,1.53,1.66 - 1.53
334,2026-04-25 16:30:00,Arsenal,Newcastle,1,14,34,1.32,1.75,1.32 - 1.75
335,2026-04-27 19:00:00,Man United,Brentford,3,7,34,1.58,1.72,1.58 - 1.72
336,2026-05-01 19:00:00,Leeds,Burnley,15,19,34,1.19,1.61,1.19 - 1.61
337,2026-05-02 14:00:00,Brentford,West Ham,7,17,35,1.29,1.65,1.29 - 1.65
338,2026-05-02 14:00:00,Newcastle,Brighton,14,9,35,1.78,1.74,1.78 - 1.74
